Preprocessing Main Data

In [10]:
# Standard Library
import re
import gc
from typing import Literal

# Third-Party Library
import pandas as pd
import nltk
import contractions
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

# NLTK Resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [11]:
# Load the dataset
file_path = "../data/main_data.csv"
df = pd.read_csv(file_path)

print('Available Columns: ', df.columns)
print('First 5 Rows:')
df.head()

Available Columns:  Index(['review', 'sentiment'], dtype='str')
First 5 Rows:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [12]:
# Normalize

# Normalize Contractions (simply she's = she is)
def normalize_contractions(text):
    return contractions.fix(str(text))

df['normalized'] = df['review'].apply(normalize_contractions)

# Normalize Text
def normalize_text(text):
    # Remove Tag HTML
    text = re.sub(r'<.*?>', '', text)
    # Lowercase Text
    text = text.lower()
    # Remove symbols & numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove repeated characters (Elongation)
    text = re.sub(r'(.)\1{2,}', r'\1', text) 
    
    return text

df['normalized'] = df['normalized'].apply(normalize_text)

In [13]:
# Tokenization

df['tokens'] = df['normalized'].apply(lambda x: word_tokenize(x))

df[['normalized', 'tokens']].head(5)


,normalized,tokens
0,one of the other reviewers has mentioned that ...,"[one, of, the, other, reviewers, has, mentione..."
1,a wonderful little production the filming tech...,"[a, wonderful, little, production, the, filmin..."
2,i thought this was a wonderful way to spend ti...,"[i, thought, this, was, a, wonderful, way, to,..."
3,basically there is a family where a little boy...,"[basically, there, is, a, family, where, a, li..."
4,petter matteis love in the time of money is a ...,"[petter, matteis, love, in, the, time, of, mon..."


In [14]:
# Stopword (remove common words)

stop_words = set(stopwords.words('english'))
def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df['no_stopwords'] = df['tokens'].apply(remove_stopwords)

df[['tokens','no_stopwords']].head(5)


,tokens,no_stopwords
0,"[one, of, the, other, reviewers, has, mentione...","[one, reviewers, mentioned, watching, oz, epis..."
1,"[a, wonderful, little, production, the, filmin...","[wonderful, little, production, filming, techn..."
2,"[i, thought, this, was, a, wonderful, way, to,...","[thought, wonderful, way, spend, time, hot, su..."
3,"[basically, there, is, a, family, where, a, li...","[basically, family, little, boy, jake, thinks,..."
4,"[petter, matteis, love, in, the, time, of, mon...","[petter, matteis, love, time, money, visually,..."


In [15]:
# Text Reduction (Stemmization/Lemmatization)
type ReductionMethod = Literal['stemmization', 'lemmatization', 'both']
reduction_method: ReductionMethod = 'lemmatization'

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def apply_text_reduction(tokens: list, method: Literal['stemmization', 'lemmatization', 'both']) -> list:
    if method == 'stemmization':
        return [stemmer.stem(word) for word in tokens]
    elif method == 'lemmatization':
        return [lemmatizer.lemmatize(word) for word in tokens]
    elif method == 'both':
        stemmed = [stemmer.stem(word) for word in tokens]
        lemmatized = [lemmatizer.lemmatize(stem) for stem in stemmed]
        return lemmatized
    else:
        raise ValueError("Wrong Parameter Needed")

df['processed_tokens'] = df['no_stopwords'].apply(lambda tokens: apply_text_reduction(tokens, method=reduction_method))

df[['no_stopwords', 'processed_tokens']].head()


,no_stopwords,processed_tokens
0,"[one, reviewers, mentioned, watching, oz, epis...","[one, reviewer, mentioned, watching, oz, episo..."
1,"[wonderful, little, production, filming, techn...","[wonderful, little, production, filming, techn..."
2,"[thought, wonderful, way, spend, time, hot, su...","[thought, wonderful, way, spend, time, hot, su..."
3,"[basically, family, little, boy, jake, thinks,...","[basically, family, little, boy, jake, think, ..."
4,"[petter, matteis, love, time, money, visually,...","[petter, matteis, love, time, money, visually,..."


In [16]:
# Generating Processed Text (from processed tokens)
# For TF-IDF Vectorization
df['processed_text'] = df['processed_tokens'].apply(lambda tokens: ' '.join(tokens))

df[['processed_tokens','processed_text']].head()

,processed_tokens,processed_text
0,"[one, reviewer, mentioned, watching, oz, episo...",one reviewer mentioned watching oz episode hoo...
1,"[wonderful, little, production, filming, techn...",wonderful little production filming technique ...
2,"[thought, wonderful, way, spend, time, hot, su...",thought wonderful way spend time hot summer we...
3,"[basically, family, little, boy, jake, think, ...",basically family little boy jake think zombie ...
4,"[petter, matteis, love, time, money, visually,...",petter matteis love time money visually stunni...


In [17]:
# Generate Dataframe into [id_data, sentiment_label, review, processed_text, processed_tokens]
df_generated = pd.DataFrame({
    'id_data': range(1, len(df) + 1),
    'sentiment_label': df['sentiment'],
    'review': df['review'],
    'processed_text': df['processed_text'],
    'processed_tokens': df['processed_tokens'],
})

df_generated.head()

,id_data,sentiment_label,review,processed_text,processed_tokens
0,1,positive,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode hoo...,"[one, reviewer, mentioned, watching, oz, episo..."
1,2,positive,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...,"[wonderful, little, production, filming, techn..."
2,3,positive,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...,"[thought, wonderful, way, spend, time, hot, su..."
3,4,negative,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...,"[basically, family, little, boy, jake, think, ..."
4,5,positive,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...,"[petter, matteis, love, time, money, visually,..."


In [ ]:
# Export Preprocessed Data to CSV
output_file_path = '../data/preprocessed_data.csv'

# Export dataframe to CSV
df_generated.to_csv(output_file_path, index=False)

print(f"Success export to: {output_file_path}")


Success export to: ../data/preprocessed_data.csv
